# UseItUp — Explainable Recipe Recommendation

**CS 4580/5580 Final Project** | Joseph Yu · Gavin Onghai · Helen Mao

---

UseItUp turns your pantry into a personalised meal plan and explains every decision in plain
language. Enter the ingredients you have on hand, choose your dietary goals and time budget,
then press **Recommend**. The system runs three stages:

1. **Stage 1 — Matching & Filtering:** scores every recipe by ingredient overlap and eliminates
   those that violate your hard dietary constraints (allergies, dietary restrictions).
2. **Stage 2 — Case-Based Reasoning:** retrieves the most similar recipe to your highest-rated
   past meals and adapts it to meet your current goals via ingredient substitution.
3. **Stage 3 — Explanation:** generates a goal trace, a counterfactual, a CBR lineage trace,
   and an ingredient-utilisation report.

Every cell is **idempotent** — re-run any cell at any time and the single `state` dict keeps
everything consistent. Work through cells in order on first use.

In [1]:
%matplotlib inline

import sys
from pathlib import Path
from collections import Counter

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import ipywidgets as widgets
import pandas as pd
from IPython.display import display, Markdown, HTML, clear_output

# ── Locate project root (works from notebooks/ or project root) ──────────────
_here = Path().resolve()
_root = _here.parent if _here.name == "notebooks" else _here
if str(_root / "src") not in sys.path:
    sys.path.insert(0, str(_root / "src"))

from useitup.pipeline import recommend, Recommendation
from useitup.profile import load_profile, SoftPreferences
from useitup.data_loader import load_recipes
from useitup.explain import render_explanation

# ── Load the curated 102-recipe corpus (much more variety than the 10-recipe sample) ──
RECIPES      = load_recipes(_root / "data" / "recipes_curated.json")
BASE_PROFILE = load_profile("demo_user", base_dir=_root / "data" / "profiles")

# ── Single mutable state dict ────────────────────────────────────────────────
state: dict = {
    "profile":   BASE_PROFILE,
    "results":   None,
    "out_main":  widgets.Output(),
    "out_trace": widgets.Output(),
    "out_alt":   widgets.Output(),
}

# ── Rendering helpers ────────────────────────────────────────────────────────
def _recipe_card_md(rec, adaptations=None) -> str:
    stars = "★" * rec.difficulty + "☆" * (5 - rec.difficulty)
    header = (
        f"## 🍽 {rec.name}\n\n"
        f"**Cuisine:** {rec.cuisine} &nbsp;·&nbsp; "
        f"**Prep:** {rec.prep_time_min} min &nbsp;·&nbsp; "
        f"**Difficulty:** {stars}\n"
    )
    sub_set = {a.original.lower() for a in (adaptations or [])}
    ing_lines = []
    for i in rec.ingredients:
        qty = f" — {i.quantity} {i.unit}".rstrip() if i.quantity else ""
        marker = " 🔄" if i.name.lower() in sub_set else ""
        ing_lines.append(f"- {i.name}{qty}{marker}")
    step_lines = "\n".join(f"{n}. {s}" for n, s in enumerate(rec.instructions, 1))
    return f"{header}\n### Ingredients\n" + "\n".join(ing_lines) + f"\n\n### Instructions\n{step_lines}"


def _refresh_outputs(results=None) -> None:
    """Populate all three Output widgets from results (or state['results'])."""
    if results is None:
        results = state["results"]
    if not results:
        return
    r: Recommendation = results[0]
    recipe = r.adapted_recipe.recipe
    soft_map = {sr.recipe.id: sr.soft_score for sr in r.filter_result.survivors}

    # ── Recipe card + explanation ─────────────────────────────────────────────
    with state["out_main"]:
        clear_output(wait=True)
        display(Markdown(_recipe_card_md(recipe, r.adapted_recipe.adaptations)))
        display(Markdown("---\n" + render_explanation(r.explanation)))

    # ── Decision trace bar chart + top-5 table ────────────────────────────────
    with state["out_trace"]:
        clear_output(wait=True)
        elim = Counter(e.rule_name for e in r.decision_log if not e.passed)
        if elim:
            fig, ax = plt.subplots(figsize=(7, 3))
            rules  = list(elim.keys())
            counts = [elim[k] for k in rules]
            colors = [
                "#c0392b"
                if any(kw in k for kw in ("Allergy", "Dietary", "Pantry")) else "#e67e22"
                for k in rules
            ]
            ax.barh(rules, counts, color=colors, edgecolor="white")
            ax.set_xlabel("Recipes Eliminated")
            ax.set_title("Stage 1 — Decision Log: Eliminations per Rule")
            ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
            plt.tight_layout()
            plt.show()
            plt.close()
        rows = [
            {
                "Recipe":     rec.cbr_match.recipe.name,
                "Cuisine":    rec.cbr_match.recipe.cuisine,
                "CBR Sim":    round(rec.cbr_match.similarity_score, 3),
                "Soft Score": round(soft_map.get(rec.cbr_match.recipe.id, 0.0), 3),
                "Prep (min)": rec.cbr_match.recipe.prep_time_min,
            }
            for rec in results[:5]
        ]
        df = pd.DataFrame(rows)
        display(Markdown("### Top 5 CBR Candidates"))
        display(HTML(df.to_html(index=False)))

    # ── Alt output placeholder (overwritten by scenario buttons) ─────────────
    with state["out_alt"]:
        clear_output(wait=True)
        display(Markdown(
            "*Click a scenario button in the **Alternative Scenarios** cell to see "
            "the counterfactual result here.*"
        ))


# ── Initial recommendation run ────────────────────────────────────────────────
state["results"] = recommend(state["profile"], RECIPES, top_k=5)
_refresh_outputs()
print(f"✓ {len(RECIPES)} recipes loaded.")
print(f"  Top pick: {state['results'][0].adapted_recipe.recipe.name!r}")

✓ 102 recipes loaded.
  Top pick: 'Lemon Herb Chicken Orzo'


In [2]:
# ── Pantry items ─────────────────────────────────────────────────────────────
pantry_box = widgets.Textarea(
    value=", ".join(BASE_PROFILE.pantry),
    placeholder="Comma-separated items, e.g.: eggs, garlic, chicken breast",
    description="Pantry:",
    style={"description_width": "80px"},
    layout=widgets.Layout(width="92%", height="80px"),
)

# ── Goals (multi-select) ──────────────────────────────────────────────────────
_GOAL_OPTIONS = [
    "high_protein", "low_cost", "vegetarian", "vegan", "low_carb", "quick", "dairy_free",
]
goals_select = widgets.SelectMultiple(
    options=_GOAL_OPTIONS,
    value=list(BASE_PROFILE.soft_preferences.goals),
    description="Goals:",
    style={"description_width": "80px"},
    layout=widgets.Layout(height="140px", width="92%"),
)

# ── Max prep-time slider ──────────────────────────────────────────────────────
_default_prep = BASE_PROFILE.soft_preferences.max_prep_time_min or 45
prep_slider = widgets.IntSlider(
    value=_default_prep,
    min=10, max=120, step=5,
    description="Max prep (min):",
    style={"description_width": "120px"},
    layout=widgets.Layout(width="92%"),
    readout_format="d",
)

# ── Recommend button ──────────────────────────────────────────────────────────
btn_recommend = widgets.Button(
    description="  Recommend",
    button_style="success",
    icon="cutlery",
    layout=widgets.Layout(width="180px", height="36px"),
)
status_lbl = widgets.Label(value="")


def _on_recommend(_btn):
    status_lbl.value = "⏳ Running pipeline …"
    pantry = [p.strip() for p in pantry_box.value.split(",") if p.strip()]
    new_prefs = BASE_PROFILE.soft_preferences.model_copy(update={
        "goals": list(goals_select.value),
        "max_prep_time_min": prep_slider.value,
    })
    new_profile = BASE_PROFILE.model_copy(update={
        "pantry": pantry,
        "soft_preferences": new_prefs,
    })
    state["profile"] = new_profile
    try:
        state["results"] = recommend(new_profile, RECIPES, top_k=5)
        _refresh_outputs()
        name = state["results"][0].adapted_recipe.recipe.name
        status_lbl.value = f"✓ Recommended: {name!r}"
    except ValueError as exc:
        status_lbl.value = f"✗ {exc}"


btn_recommend.on_click(_on_recommend)

display(widgets.VBox([
    widgets.Label("Select pantry ingredients, goals, and max prep time, then click Recommend:"),
    pantry_box,
    goals_select,
    prep_slider,
    widgets.HBox([btn_recommend, status_lbl]),
]))

In [8]:
# Recipe card + full explanation — updates on every Recommend click.
# Re-running this cell re-displays the current recommendation.
display(state["out_main"])

Output()

In [9]:
# Decision-log bar chart (Stage 1 eliminations) + top-5 CBR candidates table.
# Re-running re-displays the current trace.
display(state["out_trace"])

Output()

In [10]:
# ── Counterfactual scenarios ────────────────────────────────────────────────
# Each button mutates the *current* profile in one targeted way and re-runs the
# pipeline. The output panel shows a side-by-side comparison: Original ↔ Alt,
# so you can see exactly what the change flipped.

def _adapt_summary(adaptations) -> str:
    if not adaptations:
        return "_(no ingredient substitutions)_"
    return "  ".join(f"`{a.original}`→`{a.replacement}`" for a in adaptations)


def _comparison_table(orig, alt, label: str) -> str:
    o_recipe = orig.adapted_recipe.recipe
    a_recipe = alt.adapted_recipe.recipe
    flipped = "🔁 **Recommendation flipped**" if o_recipe.id != a_recipe.id else "↪ **Same recipe** (re-ranked or re-adapted)"
    rows = [
        ("Recipe",        o_recipe.name, a_recipe.name),
        ("Cuisine",       o_recipe.cuisine, a_recipe.cuisine),
        ("Prep time",     f"{o_recipe.prep_time_min} min", f"{a_recipe.prep_time_min} min"),
        ("CBR similarity", f"{orig.cbr_match.similarity_score:.2f}", f"{alt.cbr_match.similarity_score:.2f}"),
        ("Adaptations",
            _adapt_summary(orig.adapted_recipe.adaptations),
            _adapt_summary(alt.adapted_recipe.adaptations)),
    ]
    body = "\n".join(f"| **{k}** | {o} | {a} |" for k, o, a in rows)
    return (
        f"### ↔ Scenario: _{label}_  \n{flipped}\n\n"
        "| | Original | Counterfactual |\n|---|---|---|\n" + body
    )


def _run_scenario(label: str, hc_override=None, prefs_kwargs=None, pantry_override=None) -> None:
    """Re-run pipeline with a one-axis change; show side-by-side vs. baseline."""
    base = state["profile"]
    hc = hc_override if hc_override is not None else base.hard_constraints
    extra = prefs_kwargs or {}
    new_prefs = base.soft_preferences.model_copy(update=extra)
    new_pantry = pantry_override if pantry_override is not None else base.pantry
    new_profile = base.model_copy(update={
        "hard_constraints": hc,
        "soft_preferences": new_prefs,
        "pantry": new_pantry,
    })
    with state["out_alt"]:
        clear_output(wait=True)
        try:
            alt = recommend(new_profile, RECIPES, top_k=5)
        except ValueError as exc:
            display(Markdown(f"### ↔ Scenario: _{label}_\n\n**No survivors after the change:** {exc}"))
            return
        orig = state["results"][0]
        display(Markdown(_comparison_table(orig, alt[0], label)))
        display(Markdown("---"))
        display(Markdown(_recipe_card_md(alt[0].adapted_recipe.recipe, alt[0].adapted_recipe.adaptations)))
        display(Markdown("---\n" + render_explanation(alt[0].explanation)))


# ── Scenario buttons (six diverse flips) ─────────────────────────────────────
_BTN_W = "260px"

btn_no_gluten = widgets.Button(
    description="Drop gluten-free constraint",
    button_style="info", layout=widgets.Layout(width=_BTN_W),
)
btn_vegetarian = widgets.Button(
    description="Go vegetarian (substitute proteins)",
    button_style="warning", layout=widgets.Layout(width=_BTN_W),
)
btn_vegan = widgets.Button(
    description="Go vegan (heavy adaptation)",
    button_style="warning", layout=widgets.Layout(width=_BTN_W),
)
btn_eu = widgets.Button(
    description="Switch taste → Italian/French",
    button_style="info", layout=widgets.Layout(width=_BTN_W),
)
btn_allergies = widgets.Button(
    description="Add nut + dairy allergies",
    button_style="danger", layout=widgets.Layout(width=_BTN_W),
)
btn_minimal = widgets.Button(
    description="Empty pantry except staples",
    button_style="info", layout=widgets.Layout(width=_BTN_W),
)


btn_no_gluten.on_click(lambda _b: _run_scenario(
    "Drop gluten-free constraint",
    hc_override=[c for c in state["profile"].hard_constraints if c != "gluten-free"],
))
btn_vegetarian.on_click(lambda _b: _run_scenario(
    "Vegetarian goal added — meat substitutions fire",
    prefs_kwargs={"goals": list(set(state["profile"].soft_preferences.goals) | {"vegetarian"})},
))
btn_vegan.on_click(lambda _b: _run_scenario(
    "Vegan goal — meat AND dairy get substituted",
    prefs_kwargs={"goals": ["vegan", "high_protein"]},
))
btn_eu.on_click(lambda _b: _run_scenario(
    "Preferred cuisines → Italian + French",
    prefs_kwargs={"preferred_cuisines": ["Italian", "French"]},
))
btn_allergies.on_click(lambda _b: _run_scenario(
    "Hard constraints + nut-free + dairy-free",
    hc_override=list(set(state["profile"].hard_constraints) | {"nut-free", "dairy-free"}),
))
btn_minimal.on_click(lambda _b: _run_scenario(
    "Bare pantry: only oil, garlic, onion, salt",
    pantry_override=["olive oil", "garlic", "onion", "salt", "pepper", "egg"],
))


display(widgets.VBox([
    widgets.Label("Live counterfactual demos — each button changes ONE axis of the profile:"),
    widgets.HBox([btn_no_gluten, btn_vegetarian, btn_vegan]),
    widgets.HBox([btn_eu, btn_allergies, btn_minimal]),
    widgets.Label("↓ Side-by-side comparison appears below ↓"),
    state["out_alt"],
]))